<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/19_Bagging_Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!gdown 1BNSKS0Lg5S4-YWz3_TKXxq6ngC_p98U9
!gdown 1yGyOy2-QueXEspQgPkPwcuH7v1lUJ7Fn

Downloading...
From: https://drive.google.com/uc?id=1BNSKS0Lg5S4-YWz3_TKXxq6ngC_p98U9
To: /content/hotel_bookings_5k.csv
100% 344k/344k [00:00<00:00, 30.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yGyOy2-QueXEspQgPkPwcuH7v1lUJ7Fn
To: /content/hotel_bookings.csv
100% 16.9M/16.9M [00:00<00:00, 60.9MB/s]


In [ ]:
import numpy as np

In [ ]:
samplings = np.random.randint(1, 400, 200)
samplings

array([ 20, 108, 265,  24, 368, 359, 370, 221, 259,  70, 378,  55, 202,
       275, 229, 283,   6, 359, 343,  26, 293, 266, 388, 153, 214, 373,
       225, 264, 272, 242,  96, 174,  78,  81, 158, 362,  88,  41, 103,
       362, 398, 324, 339, 302, 353, 381,  60, 257, 143,  15, 396, 145,
       207, 342, 112,  51, 199, 147, 253, 321,  50,  57,  32,  18, 342,
       342, 239, 368, 223, 342,  39, 335, 164, 290,  69,  68,  84, 116,
       372, 203,  33, 359, 225, 168, 326, 307, 298, 266, 318,  46, 295,
       350,  78, 302,  98, 153, 179, 194, 250, 372, 241, 383, 106, 153,
       272, 322,  51, 107, 245, 268, 236, 289, 342, 320, 135, 170,  22,
       356, 146,  24, 385, 192, 176, 365, 348,  46, 397, 145, 387, 207,
       281, 297, 216, 186, 303,  30, 228, 288, 280, 162, 276,  63,  66,
       343, 177,   9, 259, 344, 374,  92, 354, 336, 117,  23, 131, 320,
       398, 369,  76, 108, 248, 369, 201, 145, 275, 383, 346, 192,  88,
       292, 223, 197, 389, 327, 165,  44,  45,  70, 139, 134, 29

In [ ]:
# 34 in samplings

False

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
bookings = pd.read_csv("hotel_bookings_5k.csv")

In [ ]:
data = pd.get_dummies(bookings, drop_first=True, dtype=int)

In [ ]:
X = data.drop(columns="is_canceled")
y = data["is_canceled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def score(model):
    model.fit(X_train, y_train)
    print("train:", round(model.score(X_train, y_train), 3),
          "| test:", round(model.score(X_test, y_test), 3))
    return model

def first_questions(model):
    roots = [X.columns[tree.tree_.feature[0]] for tree in model.estimators_]
    return pd.Series(roots).value_counts()


In [ ]:
print("baseline (everyone shows up):", round(1 - y_test.mean(), 3))

best_tree = score(DecisionTreeClassifier(max_depth=6, random_state=42))

deep_tree = score(DecisionTreeClassifier(random_state=42))

baseline (everyone shows up): 0.619
train: 0.821 | test: 0.79
train: 0.997 | test: 0.735


In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier(n_estimators=100, max_depth=9)
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)


score(rfc)

train: 0.837 | test: 0.794


RandomForestClassifier(max_depth=9)

# Boosting

In [ ]:
stump = score(DecisionTreeClassifier(max_depth=1))                                               # ONE question
bagged = score(BaggingClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=300, random_state=42))   # 300 stumps vote
boosted = score(GradientBoostingClassifier(max_depth=1, n_estimators=300, random_state=42))         # 300 stumps in a relay

train: 0.752 | test: 0.726
train: 0.752 | test: 0.726
train: 0.824 | test: 0.791


In [ ]:
guess = pd.Series(y_train.mean(), index=y_train.index)        # start: every booking gets 38 %
print("start        | average error:", round((y_train - guess).abs().mean(), 3))

for step in range(1, 10):
    error = y_train - guess                                    # what is still wrong
    fixer = DecisionTreeRegressor(max_depth=3, random_state=42)
    fixer.fit(X_train, error)                                  # a small tree learns the ERRORS
    guess = guess + 0.5 * fixer.predict(X_train)               # accept half of its correction
    print("after tree", step, "| average error:", round((y_train - guess).abs().mean(), 3))

start        | average error: 0.471
after tree 1 | average error: 0.393
after tree 2 | average error: 0.351
after tree 3 | average error: 0.33
after tree 4 | average error: 0.316
after tree 5 | average error: 0.299
after tree 6 | average error: 0.292
after tree 7 | average error: 0.283
after tree 8 | average error: 0.279
after tree 9 | average error: 0.278


In [ ]:
gbc = GradientBoostingClassifier(n_estimators=500, max_depth=2)
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)

In [ ]:
score(gbc)

train: 0.848 | test: 0.791


GradientBoostingClassifier(max_depth=2, n_estimators=500)